<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_04_sequence_model_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 04 — Sequences, and a Look at Attention

**Deep Learning for Engineering · Aalborg University · Part 1**

L5.2 covers three ways of getting information across a sequence: a recurrent
network, a gated recurrent network, and attention. This notebook builds all
three, on one small engineering forecasting problem, and ends with the honest
comparison that L5.2's closing slide asks for.

The problem: **one-step-ahead forecasting of substation demand.** Given the last
twenty-four hourly readings, predict the next one. Forty days of synthetic data,
generated on your machine, with a daily cycle, a weekly cycle and correlated
noise.

Every model here is written out rather than called. There is no `nn.LSTM` in this
notebook and no `nn.MultiheadAttention`. You will write the recurrence, the four
gates and the scaled dot product yourself, in about five lines each, because
those five lines are the entire content of the architectures and the library call
hides them.

The closing result is worth stating up front, so that you read the notebook
looking for evidence rather than for confirmation:

> On a dataset this size, a plain multilayer perceptron with 417 parameters
> matches everything else here, at a fiftieth of the compute.

That is not a criticism of recurrence or of attention. It is a statement about
when they are needed.

---

## 0 · Setup and the data

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

core.set_seed(0)

series = core.load_profile(n_days=40, seed=21)
X, y = core.make_windows(series, window=24, horizon=1)
X_train, y_train, X_test, y_test = core.split_series(X, y, frac=0.75)

print("series  :", series.shape, " (40 days x 24 hours)")
print("windows :", X.shape, " targets:", y.shape)
print("training:", X_train.shape[0], " held out:", X_test.shape[0])

core.plot_series(series[:14 * 24], title="The first two weeks")
plt.show()

**What you should see.** `series : (960,)`, `windows : (936, 24, 1)`,
`targets: (936, 1)`, 702 training windows and 234 held out, and a trace with a
clear daily rhythm and two visibly lighter days at each weekend.

Three ingredients went into that series, all of them things a real feeder has: a
daily cycle with a morning shoulder and a taller evening peak, a weekly cycle in
which weekends are about fifteen per cent lighter, and **correlated noise** — an
AR(1) process rather than independent draws, because weather and human behaviour
are correlated from hour to hour.

The correlated noise is the ingredient that makes this a forecasting problem
rather than a curve-fitting problem. Independent noise could be averaged away by
any model with enough capacity; correlated noise cannot, and the best achievable
error is set by how much of the next hour is genuinely unpredictable from the
last twenty-four.

**The split is in time order, and that is not a detail.** `core.split_series`
takes the first 75 % of the windows for training and the last 25 % for testing,
without shuffling. Shuffle first and two overlapping windows share twenty-three
of their twenty-four values, so a window in the test set is nearly identical to
one in the training set — the model has seen the future of its own test set, and
your held-out error becomes fiction. This is the single most common way a
published time-series result turns out to be worthless.

---

## 1 · Baselines before models

Two baselines, both free, both mandatory before you are allowed to be pleased
with a network.

**Persistence**: predict that the next hour equals this hour. This is the
forecast a plant operator makes without a computer, and in short-horizon
forecasting it is famously hard to beat.

**Climatology**: predict the training mean, always. This is the forecast of a
model that has learned nothing, and it is the number your $R^2$ is implicitly
measured against.

### Your turn

In [ ]:
# TODO 1 --- two baselines that need no training ---------------------------------------------
# Two `...` to replace:
#   line 1  ->  X_test[:, -1, 0]           persistence: the last value of each window
#   line 2  ->  y_train.mean()             climatology: the training mean, for every window
pred_persistence = ...                            # <- X_test[:, -1, 0]
pred_mean        = ...                            # <- y_train.mean()

mse_persistence = float(np.mean((pred_persistence - y_test[:, 0]) ** 2))
mse_mean        = float(np.mean((pred_mean - y_test[:, 0]) ** 2))
# ------------------------------------------------------------------------------

In [ ]:
print(f"persistence  MSE: {mse_persistence:.6f}")
print(f"climatology  MSE: {mse_mean:.6f}")
print(f"ratio            : {mse_mean / mse_persistence:.1f} x")
print()
print(f"held-out target standard deviation: {y_test.std():.4f} p.u.")
print(f"persistence RMSE                  : {np.sqrt(mse_persistence):.4f} p.u.")

**What you should see.** `persistence MSE: 0.003320`, `climatology MSE:
0.024710`, a ratio of about 7.4, and a persistence RMSE of about 0.058 per unit.

Persistence is seven times better than the mean. Any model that does not beat
0.00332 has learned nothing an operator did not already know, and reporting its
$R^2$ against the mean would be misleading. Quote your baselines in the report;
a bare MSE is uninterpretable.

---

## 2 · The recurrent network, written out

A recurrent network processes one time step at a time and carries a hidden state
forward:

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}\,[\mathbf{x}_t,\ \mathbf{h}_{t-1}] + \mathbf{b}\right),
\qquad \hat{y} = \mathbf{W}_{\mathrm{out}}\mathbf{h}_T + b_{\mathrm{out}}$$

That is the whole architecture. One `Linear` layer, applied once per time step,
with its own previous output concatenated onto the next input. The "unrolling"
picture from L5.2 is a picture of a `for` loop; unrolling adds no parameters.

Three consequences follow immediately, and all three are on L5.2's slides.

**The parameter count does not depend on the sequence length.** Weight sharing
again, this time across time rather than across pixels or nodes. A 24-step window
and a 2400-step window use the same weights.

**The hidden state is a bottleneck.** Everything the model remembers about the
first hour must survive twenty-three more `tanh` applications inside a
sixteen-dimensional vector. Information does not so much get forgotten as get
crowded out.

**The gradient is a product of twenty-four Jacobians.** Each is roughly the
recurrent weight matrix times a `tanh` derivative, and $\tanh' \le 1$. Multiply
twenty-four numbers each below one and you have the vanishing gradient. Multiply
twenty-four numbers each above one and you have the exploding one. This is why
the plain recurrent network was replaced.

### Your turn

In [ ]:
# TODO 2 --- the recurrent model ----------------------------------------------------------------
# One `...` to replace, inside the time loop:
#   torch.tanh(self.cell(torch.cat([x[:, t, :], h], dim=1)))
#   [x_t, h_{t-1}] is a vector of length 1 + hidden; one Linear maps it back to hidden
class SimpleRNN(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.hidden = hidden
        self.cell   = nn.Linear(1 + hidden, hidden)
        self.head   = nn.Linear(hidden, 1)

    def forward(self, x):                         # x is (batch, time, 1)
        h = torch.zeros(x.shape[0], self.hidden)
        for t in range(x.shape[1]):
            h = ...                               # <- torch.tanh(self.cell(torch.cat([x[:, t, :], h], dim=1)))
        return self.head(h)
# ------------------------------------------------------------------------------

## 3 · The LSTM cell, written out

The LSTM's answer to the vanishing gradient is a second state — the **cell state**
$c_t$ — which is updated by addition rather than by matrix multiplication, and
three sigmoid **gates** that decide what flows where. From GBC chapter 10, and
L5.2's slides:

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{W}_f[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_f)
 && \text{forget gate: what to drop from the cell}\\
\mathbf{i}_t &= \sigma(\mathbf{W}_i[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_i)
 && \text{input gate: how much of the candidate to admit}\\
\mathbf{o}_t &= \sigma(\mathbf{W}_o[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_o)
 && \text{output gate: how much of the cell to expose}\\
\mathbf{g}_t &= \tanh(\mathbf{W}_g[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_g)
 && \text{the candidate update}\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t\\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t)
\end{aligned}
$$

The line that matters is the one for $\mathbf{c}_t$. If the forget gate is near
one and the input gate near zero, then $c_t \approx c_{t-1}$ and the gradient
passes back through that step essentially unchanged — a **multiplication by
something close to one instead of by a Jacobian**. That is the whole mechanism,
and it is why the LSTM could be trained over hundreds of steps when the plain
recurrent network could not.

Note what it costs: four weight matrices where the plain cell had one, so about
four times the parameters and four times the compute per step.

In practice you would write `nn.LSTM(input_size=1, hidden_size=16,
batch_first=True)` and get a faster, fused implementation. Write it out once
first. The library version is exactly these six lines, and knowing that is the
difference between using an LSTM and quoting one.

### Your turn

In [ ]:
# TODO 3 --- the LSTM cell, written out -------------------------------------------------------
# Two `...` to replace, inside the time loop:
#   line 1  ->  f * c + i * g                 forget some of the old cell state, add some new
#   line 2  ->  o * torch.tanh(c)             the output gate reads the cell state
class SimpleLSTM(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.hidden = hidden
        self.gates  = nn.Linear(1 + hidden, 4 * hidden)   # all four gates at once
        self.head   = nn.Linear(hidden, 1)

    def forward(self, x):
        n, H = x.shape[0], self.hidden
        h = torch.zeros(n, H)
        c = torch.zeros(n, H)
        for t in range(x.shape[1]):
            z = self.gates(torch.cat([x[:, t, :], h], dim=1))
            f = torch.sigmoid(z[:, 0*H:1*H])      # forget gate
            i = torch.sigmoid(z[:, 1*H:2*H])      # input gate
            o = torch.sigmoid(z[:, 2*H:3*H])      # output gate
            g = torch.tanh(   z[:, 3*H:4*H])      # candidate cell state
            c = ...                               # <- f * c + i * g
            h = ...                               # <- o * torch.tanh(c)
        return self.head(h)
# ------------------------------------------------------------------------------

## 4 · Self-attention, written out

Attention removes the sequential bottleneck entirely. Instead of squeezing the
past through a hidden state one step at a time, every position gets a **direct
path** to every other position.

Each of the $T$ positions is embedded into a vector of width $d$, and then
projected three ways: into a **query**, a **key** and a **value**. Position $i$
compares its query with every position's key, turns the comparison into weights
with a softmax, and takes that weighted average of the values:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

Three details that are easy to skip and expensive to misunderstand.

**The $\sqrt{d}$.** Without it, the dot products grow like $d$, the softmax
saturates, and the gradient vanishes. It is a variance correction, not a
hyperparameter.

**The positional embedding.** The equation above is *permutation invariant* in
the sequence positions — exactly the property that made a graph network the right
choice in notebook 03, and exactly the wrong property here, because in a time
series the order is the information. So a learned vector per position is added to
the embeddings. Attention has to be *told* about time, whereas a recurrent
network cannot avoid knowing about it.

**The cost.** The matrix $QK^\top$ is $T \times T$. Doubling the sequence length
quadruples the compute and the memory. A recurrent network is linear in $T$. That
trade — quadratic cost for direct access, no sequential bottleneck, and
parallelism across positions — is the whole of the transformer bargain.

### Your turn

Write one single-head self-attention block and read the prediction off the last
position.

In [ ]:
# TODO 4 --- single-head self-attention -------------------------------------------------------
# Three `...` to replace, inside forward:
#   line 1  ->  (Q @ K.transpose(1, 2)) / self.d ** 0.5     scores, shape (batch, T, T)
#   line 2  ->  torch.softmax(scores, dim=-1)                attention weights, rows sum to one
#   line 3  ->  A @ V                                        weighted sum of the values
class SelfAttention(nn.Module):
    def __init__(self, d=8, T=24):
        super().__init__()
        self.d     = d
        self.embed = nn.Linear(1, d)
        self.pos   = nn.Parameter(0.1 * torch.randn(T, d))
        self.Wq    = nn.Linear(d, d, bias=False)
        self.Wk    = nn.Linear(d, d, bias=False)
        self.Wv    = nn.Linear(d, d, bias=False)
        self.head  = nn.Linear(d, 1)

    def forward(self, x, return_attention=False):
        E = self.embed(x) + self.pos              # (batch, T, d)
        Q, K, V = self.Wq(E), self.Wk(E), self.Wv(E)
        scores  = ...                             # <- (Q @ K.transpose(1, 2)) / self.d ** 0.5
        A       = ...                             # <- torch.softmax(scores, dim=-1)
        Z       = ...                             # <- A @ V
        out     = self.head(Z[:, -1, :])          # read off the last position
        return (out, A) if return_attention else out
# ------------------------------------------------------------------------------

## 5 · And the model nobody writes a paper about

A plain multilayer perceptron on the flattened 24-vector. No recurrence, no
gates, no attention. Twenty-four inputs, sixteen hidden units, one output.

It has no notion of time at all: as far as it is concerned the twenty-four
readings are twenty-four unrelated features. It cannot generalise to a different
window length, and it would be hopeless on a sequence of a thousand steps because
the input layer would have a thousand weights per hidden unit.

On *this* problem, none of that matters. Include it, and let the numbers speak.

---

## 6 · Train all four

Same data, same optimiser, same learning rate, same loss. The epoch counts differ
because the models converge at different rates; each has been given enough to
converge, and you should check the loss curves rather than take that on trust.

### Your turn

In [ ]:
# TODO 5 --- train all four models ------------------------------------------------------------
# Five `...` to replace:
#   line 1  ->  loss_fn(model(Xt), yt)                       the training loss, each epoch
#   line 2  ->  nn.Sequential(nn.Flatten(), nn.Linear(24, 16), nn.Tanh(), nn.Linear(16, 1))
#   line 3  ->  SimpleRNN(hidden=16)
#   line 4  ->  SimpleLSTM(hidden=16)
#   line 5  ->  SelfAttention(d=8, T=24)
# The recurrent models take a minute or two each.
def train_sequence(model, epochs, lr=0.01):
    loss_fn   = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.tensor(X_train); yt = torch.tensor(y_train)
    Xv = torch.tensor(X_test);  yv = torch.tensor(y_test)
    losses, val = [], []
    start = time.time()
    for epoch in range(epochs):
        optimiser.zero_grad()
        loss = ...                                # <- loss_fn(model(Xt), yt)
        loss.backward()
        optimiser.step()
        losses.append(float(loss.item()))
        with torch.no_grad():
            val.append(float(loss_fn(model(Xv), yv).item()))
    return {"epoch": np.arange(epochs),
            "train": np.asarray(losses), "val": np.asarray(val),
            "seconds": time.time() - start}

models, histories = {}, {}
core.set_seed(0); models["perceptron"] = ...      # <- nn.Sequential(nn.Flatten(), nn.Linear(24, 16), nn.Tanh(), nn.Linear(16, 1))
histories["perceptron"] = train_sequence(models["perceptron"], 600)
core.set_seed(0); models["recurrent"] = ...       # <- SimpleRNN(hidden=16)
histories["recurrent"] = train_sequence(models["recurrent"], 300)
core.set_seed(0); models["LSTM"] = ...            # <- SimpleLSTM(hidden=16)
histories["LSTM"] = train_sequence(models["LSTM"], 300)
core.set_seed(0); models["attention"] = ...       # <- SelfAttention(d=8, T=24)
histories["attention"] = train_sequence(models["attention"], 400)
# ------------------------------------------------------------------------------

In [ ]:
rows = []
for name in ("perceptron", "recurrent", "LSTM", "attention"):
    h = histories[name]
    rows.append([name, f"{core.count_parameters(models[name]):,}",
                 f"{h['train'][-1]:.6f}", f"{h['val'][-1]:.6f}",
                 f"{h['val'][-1] / mse_persistence:.2f}",
                 f"{h['seconds']:.1f}"])
rows.append(["persistence", "0", "-", f"{mse_persistence:.6f}", "1.00", "0.0"])
print(core.error_table(rows, ["model", "parameters", "train MSE",
                              "held-out MSE", "vs persistence", "seconds"]))

fig, ax = plt.subplots(figsize=(7.4, 4.4))
for i, name in enumerate(("perceptron", "recurrent", "LSTM", "attention")):
    ax.plot(histories[name]["epoch"], histories[name]["val"], lw=1.6,
            label=name)
ax.axhline(mse_persistence, color="#111111", ls="--", lw=1.3,
           label="persistence")
ax.set_yscale("log"); ax.set_xlabel("epoch")
ax.set_ylabel("held-out mean squared error")
ax.set_title("Four architectures, one small forecasting problem")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** A table close to

| model | parameters | train MSE | held-out MSE | vs persistence | seconds |
| --- | --- | --- | --- | --- | --- |
| perceptron | 417 | 0.000693 | 0.000786 | 0.24 | 0.3 |
| recurrent | 305 | 0.001179 | 0.001271 | 0.38 | 5 |
| LSTM | 1,169 | 0.000729 | 0.000793 | 0.24 | 28 |
| attention | 409 | 0.000645 | 0.000705 | 0.21 | 14 |
| persistence | 0 | - | 0.003320 | 1.00 | 0.0 |

Your wall times will differ — the ratios between them will not.

Read it in three passes.

**Every learned model beats persistence**, by a factor of between three and five.
Good: the problem is learnable and none of the models is broken.

**The plain recurrent network is the worst of the four.** With a
sixteen-dimensional hidden state and twenty-four steps to remember, it loses
information about the start of the window — which is where yesterday's value of
this hour would be, and that is the single most informative number available.
This is the vanishing-gradient story of L5.2, visible on a twenty-four-step
sequence rather than on a thousand-step one.

**The LSTM fixes it, and lands exactly where the perceptron already was.** Four
times the parameters, ninety times the wall time, and the same held-out error.

---

## 7 · Is the attention model actually better?

The table says attention is best, by about ten per cent. Before you write that
down, ask the question Ex_04 taught you to ask: **is one run evidence?**

Train the two contenders from three seeds each and look at the spread.

### Your turn

In [ ]:
# TODO 6 --- three seeds each ------------------------------------------------------------------
# Two `...` to replace, inside the loop (a FRESH model each time):
#   line 1  ->  nn.Sequential(nn.Flatten(), nn.Linear(24, 16), nn.Tanh(), nn.Linear(16, 1))
#   line 2  ->  SelfAttention(d=8, T=24)
seed_results = {"perceptron": [], "attention": []}
for seed in (0, 1, 2):
    core.set_seed(seed)
    m = ...                                       # <- nn.Sequential(nn.Flatten(), nn.Linear(24, 16), nn.Tanh(), nn.Linear(16, 1))
    seed_results["perceptron"].append(float(train_sequence(m, 600)["val"][-1]))
    core.set_seed(seed)
    m = ...                                       # <- SelfAttention(d=8, T=24)
    seed_results["attention"].append(float(train_sequence(m, 400)["val"][-1]))
# ------------------------------------------------------------------------------

In [ ]:
rows = []
for name, vals in seed_results.items():
    vals = np.asarray(vals)
    rows.append([name, " ".join(f"{v:.6f}" for v in vals),
                 f"{vals.mean():.6f}", f"{vals.min():.6f}", f"{vals.max():.6f}"])
print(core.error_table(rows, ["model", "held-out MSE, seeds 0 / 1 / 2",
                              "mean", "best", "worst"]))

fig, ax = plt.subplots(figsize=(6.2, 3.6))
for i, (name, vals) in enumerate(seed_results.items()):
    ax.plot([i] * len(vals), vals, "o", ms=10, alpha=0.75)
    ax.plot([i - 0.15, i + 0.15], [np.mean(vals)] * 2, "-", lw=2.4,
            color="#111111")
ax.set_xticks(range(len(seed_results)))
ax.set_xticklabels(list(seed_results))
ax.set_ylabel("held-out MSE")
ax.set_title("Three seeds each — the spread is the story")
ax.grid(alpha=0.25)
plt.show()

**What you should see.** Something close to

| model | held-out MSE, seeds 0 / 1 / 2 | mean | best | worst |
| --- | --- | --- | --- | --- |
| perceptron | 0.000786 0.000709 0.000771 | 0.000755 | 0.000709 | 0.000786 |
| attention | 0.000705 0.000778 0.001066 | 0.000850 | 0.000705 | 0.001066 |

**The distributions overlap completely.** The attention model's best run is
better than any perceptron run; its worst run is fifty per cent worse than any
perceptron run; and on average it is slightly *behind*. The ten per cent
advantage in section 6's table was a property of seed 0, not of the
architecture.

This is L5.2's closing slide, arrived at from your own numbers rather than
asserted:

> For the time series in this course you have thousands of samples, not billions.
> An LSTM — often a plain multilayer perceptron — will beat a transformer on this
> data. Architecture fashion is not architecture selection.

State the conclusion carefully, because the careless version is wrong. Attention
is not a worse mechanism than a dense layer. On 702 training windows of 24 steps
it has nothing to do that a dense layer cannot already do, and it pays for the
generality with more hyperparameters, more variance across seeds, and fifty times
the compute. Give it a sequence of four thousand steps, or a million training
examples, or a task where the relevant lag varies from case to case, and the
ranking reverses. **The architecture is a claim about the data, and this data
does not support the claim.**

---

## 8 · What attention learned to look at

One thing attention gives you that a dense layer does not: the matrix $A$ is a
readable statement about which positions the model used.

In [ ]:
core.set_seed(0)
attn = SelfAttention(d=8, T=24)
_ = train_sequence(attn, 400)

with torch.no_grad():
    _, A_matrix = attn(torch.tensor(X_test[:64]), return_attention=True)
A_mean = A_matrix.numpy().mean(axis=0)
last_row = A_mean[-1]

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2))
im = axes[0].imshow(A_mean, cmap="viridis")
axes[0].set_xlabel("attends to position"); axes[0].set_ylabel("position")
axes[0].set_title("mean attention matrix over 64 held-out windows")
plt.colorbar(im, ax=axes[0], shrink=0.85)
axes[1].bar(np.arange(24), last_row, color="#1f77b4")
axes[1].set_xlabel("position in the 24-hour window")
axes[1].set_ylabel("attention weight")
axes[1].set_title("what the output position looks at")
axes[1].grid(alpha=0.25, axis="y")
plt.show()

order = np.argsort(last_row)[::-1][:5]
print("five positions the output attends to most:")
for p in order:
    print(f"  position {p:2d}  (lag {24 - p:2d} hours)  weight {last_row[p]:.3f}")
print(f"\nuniform weight for comparison: {1/24:.3f}")

**What you should see.** A 24 by 24 heat map that is far from uniform, and a
bar chart in which a handful of positions carry several times the uniform weight
of 0.042. Which positions win varies with the seed, but they cluster near the
**start** of the window — lags of roughly twenty to twenty-four hours, which is
the same hour yesterday. That is the right answer physically: demand at 18:00
today is best predicted by demand at 18:00 yesterday, and a daily cycle is the
strongest structure in the series.

Two warnings, both of which apply every time somebody shows you an attention map.

**Attention weights are not an explanation.** They tell you which values were
averaged, not why the answer came out as it did. The value vectors $V$ are
learned too, and a small weight on a large value can matter more than a large
weight on a small one. There is a substantial literature on this, and its
conclusion is that attention is *suggestive* evidence about a model's reasoning
and not *sufficient* evidence.

**One head, one layer, and 702 training windows.** A real transformer stacks
several layers of several heads each, and the interpretable structure people
report comes from looking across all of them. Do not generalise from this
picture.

What the picture does establish, and what is worth taking away, is the mechanism:
position 23 reached position 0 in **one step**, with no intermediate state to
squeeze through. In the recurrent model that same information had to survive
twenty-three `tanh` applications. That is the whole argument for attention, and
it is visible here even though it did not pay for itself on this dataset.

---

## 9 · The forecast, looked at

Numbers first, pictures second — but never numbers only.

In [ ]:
with torch.no_grad():
    curves = {name: models[name](torch.tensor(X_test)).numpy().ravel()
              for name in ("perceptron", "LSTM", "attention")}
curves["persistence"] = X_test[:, -1, 0]

core.plot_forecast(y_test, curves, n=120,
                   title="The first five days of the held-out period")
plt.show()

resid = curves["perceptron"] - y_test.ravel()
fig, ax = plt.subplots(1, 2, figsize=(12.0, 3.4))
ax[0].hist(resid, bins=30, color="#1f77b4", alpha=0.85)
ax[0].set_xlabel("residual [p.u.]"); ax[0].set_title("perceptron residuals")
ax[0].grid(alpha=0.25)
ax[1].plot(resid[:-1], resid[1:], ".", ms=4, color="#d94f2b")
ax[1].set_xlabel("residual at hour t"); ax[1].set_ylabel("residual at hour t+1")
ax[1].set_title("are the residuals correlated?")
ax[1].grid(alpha=0.25)
plt.show()

print("residual mean %.5f, standard deviation %.5f"
      % (resid.mean(), resid.std()))
print("lag-1 autocorrelation of the residuals: %.3f"
      % np.corrcoef(resid[:-1], resid[1:])[0, 1])

**What you should see.** Forecast curves that follow the measured trace
closely and are visibly better than persistence at the turning points — the
morning rise and the evening peak, which is where persistence is always late.
Then a roughly symmetric residual histogram with a mean near zero and a standard
deviation of about 0.028 per unit, a residual scatter that leans clearly
upwards, and

```
lag-1 autocorrelation of the residuals: 0.50
```

**That number is the most useful thing in this notebook, and it is a failure.**

The autocorrelation is the check that matters and it is the one most often
skipped. If consecutive residuals were independent, the scatter would be a
formless cloud and the model would have extracted everything a 24-hour window
contains. At 0.5 it has not: whatever made the model wrong at hour $t$ is still
half there at hour $t+1$, and *by construction it was knowable* — the generator's
noise is an AR(1) process with a lag-one coefficient of 0.75, so half of next
hour's error is a deterministic function of this hour's.

Notice what did **not** tell you this. The held-out MSE did not: 0.000786 looks
excellent against persistence's 0.003320, and all four architectures agree on it,
which makes it look like a floor. The loss curves did not: they are smooth and
flat and entirely unremarkable. Only the residual check did, and it took two
lines.

What you would do next, in rough order of cost: check whether a longer window
helps; give the model the *difference* from the previous hour as an extra feature
rather than making it infer the trend; or model the residual explicitly with an
AR(1) term and forecast the sum. All three are cheap. None would have been
attempted without the diagnostic.

This is the same habit Part 2 uses everywhere, with one difference: in a
physics-informed model the residual being checked is a differential equation
rather than a forecast error, and a structured residual there means the physics
is being violated somewhere specific.

---

## 10 · Save

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb04_sequence.npz")
np.savez(path,
         mse_persistence=mse_persistence, mse_mean=mse_mean,
         names=np.array(list(models)),
         params=np.array([core.count_parameters(models[n]) for n in models]),
         val=np.array([histories[n]["val"][-1] for n in models]),
         seconds=np.array([histories[n]["seconds"] for n in models]),
         seeds_perceptron=np.asarray(seed_results["perceptron"]),
         seeds_attention=np.asarray(seed_results["attention"]),
         attention_last_row=last_row)
print("wrote", path)

**What you should see.** `wrote .../Ex05_outputs/nb04_sequence.npz`.

---

## 11 · Before you move on

Answer these here.

1. The recurrent network was the worst model here. Name the mechanism, and say
   what you would expect to happen to the ranking if the window were 240 hours
   instead of 24.
2. Attention needed a positional embedding; the recurrent network did not. Explain
   why, using the word *permutation*, and connect it to notebook 03.
3. You have four models within a factor of two of each other on held-out MSE.
   Which would you deploy on a substation controller, and which single number
   from the table decided it?
4. The residual autocorrelation was about 0.5 while the held-out MSE looked
   excellent. Explain, to somebody who reports only the MSE, what that
   combination means and why their number was not wrong — only incomplete.

---

Continue with **`Ex05_05_report.ipynb`**.

*Write your answers here.*

1.
2.
3.
4.